<a href="https://colab.research.google.com/github/langchain-samples/langsmith-studio-nb/blob/main/examples/deep_agent_in_studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Deep Agent in LangSmith Studio

Build a small Deep Agent in a notebook cell, then open [LangSmith Studio](https://docs.langchain.com/langsmith/studio)
on it and watch it call tools and write files.

Studio runs in your browser and talks to an agent server over HTTP. On Colab that server runs on
Google's machine, where your browser cannot reach it, so it needs a public URL. `start_studio()`
boots the server, opens a tunnel when it needs one, and prints the link.

**You will need two API keys:**

| Key | Used for |
| --- | --- |
| `OPENAI_API_KEY` | the agent's model calls |
| `LANGSMITH_API_KEY` | tracing, so each run shows up in LangSmith |

Get them from the [OpenAI platform](https://platform.openai.com/api-keys) and from LangSmith
under [Settings → API Keys](https://smith.langchain.com/settings/apikeys). A LangSmith key
starts with `lsv2_pt_`; it is shown exactly once, so copy it right away.

---

## 1. Store your keys

### Google Colab

Colab has a built-in secret store, and it is the only place you should put a key.

1. Click the 🔑 **key icon** in the left sidebar.
2. **Add new secret** → name it exactly `OPENAI_API_KEY` → paste the value.
3. Turn on the **Notebook access** toggle. Without it the cell below cannot read the secret.
4. Repeat for `LANGSMITH_API_KEY`.

Secrets belong to your Google account rather than to this notebook, so you only do this once.

### Your own Jupyter

There is no secret store to ask, so the keys come from the environment. Either export them before
you start Jupyter, or copy `.env.example` to `.env` beside this notebook and fill it in:

```bash
cd examples && cp .env.example .env
```

> Never commit a `.env`. `.gitignore` already covers it.

---

## 2. Install and load the secrets

`langsmith-studio-nb` pulls in `langgraph-cli[inmem]`, which is what serves the agent.

In [ ]:
%pip install -qq --progress-bar off \
  "deepagents~=0.7.8" \
  "langchain~=1.3.16" \
  "langchain-openai~=1.6.0" \
  "langsmith~=0.11.1" \
  "git+https://github.com/langchain-samples/langsmith-studio-nb.git"

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()  # your own Jupyter: a .env sitting next to this notebook

try:
    from google.colab import userdata  # Colab: the secrets you stored above
except ImportError:
    pass
else:
    for key in ("OPENAI_API_KEY", "LANGSMITH_API_KEY"):
        os.environ[key] = userdata.get(key)

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "studio-nb-example"

MODEL = "openai:gpt-5.6-luna"

print("Ready.")

Colab's `userdata` reads the secrets you stored in the sidebar, and `load_dotenv()` covers your own
Jupyter. Both write into `os.environ`, which is where the model and tracing SDKs go looking.

Nothing prints a key. A notebook's output is the part that gets saved and shared, so a secret should
never land there. `python-dotenv` needs no installing: it arrives with `langgraph-cli`, which
`langsmith-studio-nb` depends on.

---

## 3. Build the agent

`create_deep_agent` gives you a planner, a filesystem, and a subagent tool on top of whatever
tools you pass in. This one gets a single fake weather tool, so there is something to call and
something to write down without signing up for anything else.

In [ ]:
from deepagents import create_deep_agent


def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"It's always sunny in {city}!"


agent = create_deep_agent(
    model=MODEL,
    tools=[get_weather],
    system_prompt=(
        "You are a travel assistant.\n"
        "Check the weather for every city the user mentions.\n"
        "Keep your working notes in files, and write your final answer to trip.md."
    ),
)

---

## 4. Let Studio trust the tunnel

**Colab only.** Studio blocks agent servers on any host you have not told it about, and on Colab
it reaches your server through a Cloudflare tunnel. Allow the domain once:

1. Open [Studio](https://smith.langchain.com/studio/connect?mode=graph).
2. Click **Configure connection**.
3. Expand **Advanced Settings**.
4. Under **Allowed Domains**, add `*.trycloudflare.com`.
5. **Save**.

Add the wildcard, not the exact hostname Studio offers to add for you. That one changes every time
a tunnel opens. The list lives in your browser's local storage, so it is per browser and per user,
and there is no workspace-level setting. Skip all of this in your own Jupyter, because there is no
tunnel to allow.

Without it, the link in the next cell looks like it does nothing at all.

---

## 5. Open Studio

`start_studio()` serves the notebook variable named `agent` and prints a link.

In [ ]:
from langsmith_studio_nb import start_studio

start_studio()

Open the link and give it something to do:

> I'm visiting Tokyo, Lisbon, and Oslo next month. What should I pack?

Studio calls the weather tool once per city, and shows the notes and `trip.md` appearing in the
agent's filesystem as it goes. A longer task would make it plan with a todo list first. A job this
small it just does. Every run lands in your LangSmith project as well.

Changed the agent? Re-run the cell that builds it, then this one. `start_studio()` stops the
previous server first. Hot reload is not available from a notebook, so that re-run is what picks up
your edit.

---

## Notes

- **The tunnel URL is public and unauthenticated** for as long as the cell runs, and anyone holding
  the link can drive your agent. Fine for a demo like this one. Think twice before pointing it at
  anything that touches real data or spends real money. The tunnel dies with the kernel.
- **Colab reclaims idle runtimes** after about 90 minutes, and you get a fresh tunnel URL when you
  reconnect.
- **Several agents at once.** `start_studio("planner", "writer")` serves both variables over one
  tunnel and lets you switch between them in Studio's graph menu.
- **Running in your own Jupyter?** `start_studio()` skips the tunnel and serves `localhost`, since
  the browser is already on the same machine. Pass `tunnel=True` or `tunnel=False` when the guess is
  wrong.

Full API and troubleshooting: the [README](https://github.com/langchain-samples/langsmith-studio-nb#readme).